<div align="center">

# Week 06: User-Centred Design in Practice: Interviews, Personas and Tasks

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-06) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/Week06_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the twelve patients of the waiting room with their attributes and needs, and the operator times of the keystroke-level model. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


# The twelve patients of the waiting room (course data): age in years, visits a
# year, digital skill from 1 to 5, travel time in minutes and the eight themes of
# their needs, one digit per theme in the order of THEMES.
THEMES = ["finding the department", "waiting without information",
          "small text at the kiosk", "forgotten patient number",
          "booking after hours", "long journeys", "booking for a relative",
          "needs an interpreter"]
PATIENTS = pd.DataFrame([
    ("Fatma", 74, 12, 1, 20, "11100000"), ("Hasan", 69, 10, 2, 25, "10010000"),
    ("Emine", 78, 14, 1, 15, "10110000"), ("Mehmet", 71, 9, 2, 30, "01100000"),
    ("Elif", 31, 2, 5, 25, "01001000"), ("Burak", 38, 1, 4, 35, "01001000"),
    ("Zeynep", 29, 3, 5, 20, "10000000"), ("Murat", 42, 2, 4, 40, "00001000"),
    ("Ayten", 45, 6, 3, 80, "10010100"), ("Ibrahim", 52, 5, 2, 95, "10100010"),
    ("Selin", 39, 7, 4, 70, "01000110"), ("Kemal", 48, 4, 3, 85, "00000101"),
], columns=["name", "age", "visits", "skill", "travel", "themes"])
# Operators of the keystroke-level model in seconds (Card, Moran and Newell).
KLM = {"K": 0.28, "P": 1.10, "B": 0.20, "H": 0.40, "M": 1.35}
print("Setup complete. Running in Colab:", IN_COLAB, "| patients:", len(PATIENTS),
      "| themes:", len(THEMES))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 6.1": (lambda: float(np.round((52 - 50) / 10, 6)), 1e-6),
    "Exercise 6.2": (lambda: float(np.sqrt((1.2 - 0.2) ** 2 + (-0.5 - 0.7) ** 2)), 1e-6),
    "Exercise 6.3": (lambda: 3, 1e-9),
    "Exercise A": (lambda: float(sum(1 - (1 - p) ** 8 for p in [0.5, 0.25, 0.1])), 1e-6),
    "Exercise B": (lambda: float(np.sqrt(((74 - 31) / 16.59) ** 2 + ((12 - 2) / 4.04) ** 2)), 1e-6),
    "Exercise C": (lambda: 20 + 0.25 * 30 + 0.1 * 45, 1e-6),
    "Exercise D": (lambda: 3 * 1.35 + 6 * 0.28, 1e-6),
    "Exercise E": (lambda: float(STAGE6["hours saved a year"]), 1e-6),
}

## Python step 6: Scaling, distances and groups

Personas start from a table of people described by numbers, such as age, visits and digital skill. This step puts the numbers on one scale with z-scores and measures the distance between two people. It then uses the library scikit-learn &#91;[4](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#ref-4)&#93; to group people with k-means. Each tool is tried on a small example before Section 2 applies it to the patients of the waiting room &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#ref-1), [2](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#ref-2)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "KLM" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-06/"
               "NB06_users_and_tasks.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Profiles as a table

In [ ]:
# Python practice: a small example for learning the language
people = pd.DataFrame({"age": [70, 34, 47, 66],
                       "visits": [11, 2, 5, 9],
                       "skill": [1, 5, 3, 2]},
                      index=["P1", "P2", "P3", "P4"])
print(people)
print("means:", people.mean().round(2).to_dict())
print("standard deviations:", people.std(ddof=0).round(2).to_dict())

Each row describes one person and each column one attribute. The attributes have very different ranges: Ages vary by tens of years, while digital skill varies only from 1 to 5. A distance computed on the raw numbers would therefore be ruled by age.

### Scaling with z-scores

In [ ]:
# Python practice: a small example for learning the language
z = (people - people.mean()) / people.std(ddof=0)
print(z.round(2))
print("means after scaling:", z.mean().round(6).to_dict())

A *z-score* subtracts the mean of a column and divides by its standard deviation. Every column then has the mean 0 and the standard deviation 1, so one unit means the same in every column. The argument `ddof=0` divides by $n$, as scikit-learn does.

### Distances between people

In [ ]:
# Python practice: a small example for learning the language
d12 = np.sqrt(((z.loc["P1"] - z.loc["P2"]) ** 2).sum())
d14 = np.sqrt(((z.loc["P1"] - z.loc["P4"]) ** 2).sum())
print(f"distance P1-P2: {d12:.2f}, distance P1-P4: {d14:.2f}")
from sklearn.metrics import pairwise_distances
print(pairwise_distances(z).round(2))

The *Euclidean distance* is the square root of the sum of squared differences over all columns. P1 and P4, two older frequent visitors, are close, while P1 and P2 are far apart. `pairwise_distances` from scikit-learn computes the distances of all pairs at once.

### k-means with scikit-learn

In [ ]:
# Python practice: a small example for learning the language
from sklearn.cluster import KMeans
km = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE).fit(z)
print("group of each person:", km.labels_)
print("within-group sum of squares:", round(km.inertia_, 3))
print(people.groupby(km.labels_).mean().round(1))

`KMeans(n_clusters=2)` looks for two groups whose members lie close to the centre of their group. The argument `n_init=10` repeats the search from ten random starts and keeps the best one. The attribute `labels_` gives the group of each row, and `inertia_` the sum of the squared distances to the centres.

**Quick check.** Why must the columns be scaled before k-means?

<details><summary>Answer</summary>

Distances add squared differences, so the column with the largest range would decide the groups alone. After scaling, every attribute weighs the same.

</details>

### Exercises for Python step 6

**Exercise 6.1.** A column has the mean 50 and the standard deviation 10. Store the z-score of the value 52 in `z_52`.

In [ ]:
z_52 = None   # your answer

check("Exercise 6.1", z_52)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
z_52 = (52 - 50) / 10

check("Exercise 6.1", z_52)

**Exercise 6.2.** Two people have the z-scores (1.2, -0.5) and (0.2, 0.7). Store their Euclidean distance in `dist_ab`.

In [ ]:
dist_ab = None   # your answer

check("Exercise 6.2", dist_ab)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
dist_ab = np.sqrt((1.2 - 0.2) ** 2 + (-0.5 - 0.7) ** 2)

check("Exercise 6.2", dist_ab)

**Exercise 6.3.** Run k-means with `n_clusters=3` on `z` and store the number of different labels in `n_groups`.

In [ ]:
n_groups = None   # your answer

check("Exercise 6.3", n_groups)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
km3 = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(z)
n_groups = len(set(km3.labels_))

check("Exercise 6.3", n_groups)

## 1. Interviews: Your saturation curve &#91;[7](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#ref-7)&#93;

*Lecture: [Interviews and saturation](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#interviews)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "km" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-06/"
               "NB06_users_and_tasks.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Each interview reveals some needs, and later interviews reveal fewer new ones. Python counts the new themes of every interview and draws the saturation curve. It also predicts the curve from the share of patients who have each need, which shows why rare needs are missed.

**From the formulas to the code.** The share of patients with need $k$, $p_k$, is `p`, and the number of interviews $n$ is `n`. The expected number of themes found $\sum_k 1 - (1 - p_k)^n$ is `expected_themes(p, n)`, and the rule of Francis and colleagues is `stopping_point(new, initial, run)`.

In [ ]:
def expected_themes(p, n):
    """Expected number of themes found in n interviews, for shares p."""
    p = np.asarray(p, float)
    return float(np.sum(1 - (1 - p) ** n))

def stopping_point(new, initial=4, run=2):
    """First interview after the initial sample that closes a run of
    interviews without new themes, or None."""
    quiet = 0
    for i, k in enumerate(new, start=1):
        quiet = quiet + 1 if k == 0 else 0
        if i >= initial and quiet >= run:
            return i
    return None

print("worked example:", round(expected_themes([0.6, 0.4, 0.3, 0.2, 0.1], 6), 3))
need = math.log(0.1) / math.log(1 - 0.1)
print(f"a need of 10 percent is found with probability 0.9 after {need:.1f}",
      "interviews")
M_themes = np.array([[int(c) for c in s] for s in PATIENTS["themes"]])
prevalence = M_themes.mean(axis=0)
print(pd.Series(prevalence.round(3), index=THEMES))

**Python note.** `stopping_point` walks through the counts of new themes with `enumerate(new, start=1)`, which numbers the interviews from 1. The counter `quiet` restarts at zero whenever an interview brings a new theme.

**Exercise A.** Three needs have the shares 0.5, 0.25 and 0.1 among the patients. Store the expected number of themes found in eight interviews in `themes_8`.

In [ ]:
themes_8 = None   # your answer

check("Exercise A", themes_8)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
themes_8 = expected_themes([0.5, 0.25, 0.1], 8)

check("Exercise A", themes_8)

Part A of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_INTERVIEWS`. Left empty, the variable makes the code use a simulated interviewer who chooses patients at random and mostly asks open questions.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_INTERVIEWS = """
"""

def simulated_interviews(n=8, seed=RANDOM_STATE):
    """An interviewer who picks n patients at random; 3 in 4 questions open."""
    rng = np.random.default_rng(seed)
    rows = []
    for i, k in enumerate(rng.permutation(len(PATIENTS))[:n], start=1):
        style = rng.choice(["open", "closed", "leading"], p=[0.75, 0.15, 0.10])
        found = M_themes[k].copy()
        if style == "closed":
            first = np.flatnonzero(found)[:1]
            found = np.zeros_like(found)
            found[first] = 1
        if style == "leading":
            found = np.zeros_like(found)
            found[2] = M_themes[k][2]
        rows.append([i, PATIENTS["name"][k], style]
                    + found.tolist() + [int(style == "leading")])
    cols = ["interview", "patient", "style"] + [f"t{j + 1}" for j in range(8)]
    return pd.DataFrame(rows, columns=cols + ["biased"])

if MY_INTERVIEWS.strip():
    interviews = pd.read_csv(io.StringIO(MY_INTERVIEWS.strip()))
    source = "your own interviews from the interactive lab"
else:
    interviews = simulated_interviews()
    source = "a simulated interviewer"
print("Data:", source, "|", len(interviews), "interviews")
F = interviews[[f"t{j + 1}" for j in range(8)]].to_numpy()
seen = np.maximum.accumulate(F, axis=0)
cumulative = seen.sum(axis=1)
new = np.diff(np.concatenate([[0], cumulative]))
print(interviews[["interview", "patient", "style"]].assign(new=new,
                                                            total=cumulative))
print("stopping point (initial 4, run of 2):", stopping_point(new))
print("themes never heard:", [t for t, s in zip(THEMES, seen[-1]) if not s])

**Python note.** `np.maximum.accumulate` keeps, row after row, the largest value seen so far in each column, so a theme stays found once an interview has revealed it. `np.diff` turns the running total into the number of new themes of each interview.

In [ ]:
# Figure: saturation curve of the interviews
n_int = np.arange(1, len(interviews) + 1)
model = [expected_themes(prevalence, k) for k in n_int]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].step(n_int, cumulative, where="mid", color="#C98510", label="your interviews")
ax[0].plot(n_int, model, "o--", color="#5A4BB5", ms=4,
           label="model, patients chosen at random")
ax[0].axhline(len(THEMES), ls=":", color="#585E6E")
ax[0].set_xlabel("interview")
ax[0].set_ylabel("themes found")
ax[0].legend(loc="lower right")
by_style = interviews.assign(themes=F.sum(axis=1)).groupby("style")["themes"].mean()
ax[1].bar(by_style.index, by_style.values, color="#0E7A78")
ax[1].set_ylabel("themes per interview")
ax[1].set_xlabel("kind of question")
plt.tight_layout()
plt.show()

**Python note.** `step` draws the running total as stairs, since themes are found in whole numbers. The right panel averages the themes of each interview by the kind of question that opened it.

**Observe and reflect.** After how many interviews did you stop hearing new themes? Which needs are missing, and which patients would you have had to choose to hear them?

## 2. Personas from data: Grouping the patients &#91;[11](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#ref-11)&#93;

*Lecture: [Personas](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#personas)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "cumulative" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-06/"
               "NB06_users_and_tasks.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A persona should stand for a group of real users, not for the first person that comes to mind. Python scales the profiles of the patients, groups them with k-means and compares the groups with the ones you formed in the interactive lab. The centre of each group becomes the skeleton of a persona.

**From the formulas to the code.** The z-score $z = (x - \bar{x})/s$ is computed by `scale(df)`, the distance $d_{ij}$ by `pairwise_distances`, and the within-group sum of squares $W = \sum_k \sum_{i \in C_k} \lVert z_i - \mu_k \rVert^2$ by `within(z, labels)`.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, pairwise_distances

COLS = ["age", "visits", "skill", "travel"]

def scale(df):
    """z-scores of each column, with the standard deviation of the column."""
    return (df - df.mean()) / df.std(ddof=0)

def within(z, labels):
    """Sum of squared distances of the rows to the mean of their group."""
    z = np.asarray(z, float)
    labels = np.asarray(labels)
    return float(sum(((z[labels == g] - z[labels == g].mean(axis=0)) ** 2).sum()
                     for g in np.unique(labels)))

Z = scale(PATIENTS[COLS])
print("standard deviations:", PATIENTS[COLS].std(ddof=0).round(1).to_dict())
print("worked example, Fatma and Elif:",
      round(float(pairwise_distances(Z.iloc[[0, 4]])[0, 1]), 2))
km = KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE).fit(Z)
km_labels = km.labels_
print("k-means groups:", km_labels, "| W =", round(within(Z, km_labels), 2))

**Python note.** The functions repeat what the Python step showed, but for all twelve patients and four attributes. `iloc[[0, 4]]` selects the first and the fifth row, Fatma and Elif.

**Exercise B.** Fatma is 74 years old with 12 visits, and Elif 31 years old with 2 visits. With the standard deviations 16.59 years and 4.04 visits, store their distance on these two attributes in `d_fe`.

In [ ]:
d_fe = None   # your answer

check("Exercise B", d_fe)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
d_fe = np.sqrt(((74 - 31) / 16.59) ** 2 + ((12 - 2) / 4.04) ** 2)

check("Exercise B", d_fe)

Part B of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_GROUPS`. Left empty, the variable makes the code use a grouping by age alone, a common first guess.

In [ ]:
# Paste the results of lab Part B between the triple quotes.
MY_GROUPS = """
"""

if MY_GROUPS.strip():
    groups = pd.read_csv(io.StringIO(MY_GROUPS.strip()))
    mine = groups.set_index("patient").loc[PATIENTS["name"], "group"].to_numpy()
    source = "your own groups from the interactive lab"
else:
    mine = np.digitize(PATIENTS["age"], [40, 60])
    source = "groups by age alone"
print("Data:", source)
print(f"W of your groups: {within(Z, mine):.2f} | W of k-means: "
      f"{within(Z, km_labels):.2f}")
print("agreement with k-means (adjusted Rand index):",
      round(adjusted_rand_score(km_labels, mine), 3))
personas = PATIENTS.groupby(km_labels)[COLS].mean().round(1)
personas["members"] = PATIENTS.groupby(km_labels)["name"].apply(", ".join)
print(personas.to_string())

**Python note.** `set_index("patient").loc[...]` puts your groups in the order of the table of patients. The adjusted Rand index is 1 when two groupings agree completely, whatever numbers the groups carry, and about 0 for groupings that agree only by chance.

In [ ]:
# Figure: the patients on two scaled attributes, coloured by group
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for a, lab, title in [(ax[0], mine, "your groups"), (ax[1], km_labels, "k-means")]:
    a.scatter(Z["age"], Z["skill"], c=lab, cmap="viridis", s=70)
    for name, x, y in zip(PATIENTS["name"], Z["age"], Z["skill"]):
        a.annotate(name, (x, y), xytext=(4, 4), textcoords="offset points",
                   fontsize=8)
    a.set_xlabel("age (z-score)")
    a.set_title(f"{title}: W = {within(Z, lab):.1f}")
ax[0].set_ylabel("digital skill (z-score)")
plt.tight_layout()
plt.show()

**Python note.** Each panel shows only two of the four attributes, so groups that look mixed here can still be close in visits and travel. The title of each panel carries its within-group sum of squares.

**Observe and reflect.** Which patients did you place differently from k-means, and why? Describe the three personas in one sentence each, from the means of their groups.

## 3. Task analysis: Plans with branches &#91;[12](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#ref-12)&#93;

*Lecture: [Hierarchical task analysis](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#task-analysis)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "km_labels" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-06/"
               "NB06_users_and_tasks.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A hierarchical task analysis breaks a goal into subtasks and records the plan that orders them. When a plan contains a branch, such as choosing another day when no time is free, the time of the task depends on how often the branch is taken. Python walks through the tree and adds the times of all steps, weighted by the probability of each branch.

**From the formulas to the code.** The time of a step $t_j$ is the key `time`, and the probability of an optional step $q_j$ is the key `p`. The expected time of a task $E[T] = \sum_j q_j t_j$ is `expected_time(task)`, with $q_j = 1$ for steps that are always done.

In [ ]:
def expected_time(task):
    """Expected time of a task tree: own time plus the weighted subtasks."""
    total = task.get("time", 0.0)
    for sub in task.get("subtasks", []):
        total += sub.get("p", 1.0) * expected_time(sub)
    return total

def steps(task, depth=0):
    """Print the tree with its numbers, probabilities and expected times."""
    p = task.get("p", 1.0)
    print("  " * depth + f"{task['name']}  (p = {p:g}, E = "
          f"{expected_time(task):.1f} s)")
    for sub in task.get("subtasks", []):
        steps(sub, depth + 1)

def book_tree(q_day=0.25, p_help=0.05):
    """The booking of the lecture, with the probabilities of its two branches."""
    return {"name": "0 Book an appointment", "subtasks": [
        {"name": "1 Log in", "time": 20},
        {"name": "2 Choose a department and a time", "subtasks": [
            {"name": "2.1 Choose the department", "time": 15},
            {"name": "2.2 Choose a time", "time": 12},
            {"name": "2.3 Choose another day", "time": 18, "p": q_day}]},
        {"name": "3 Confirm", "time": 6},
        {"name": "4 Ask for help at the desk", "time": 120, "p": p_help}]}

BOOK = book_tree()
steps(BOOK)
print(f"expected time of the booking: {expected_time(BOOK):.1f} s")

**Python note.** `expected_time` calls itself for every subtask, a technique called *recursion*. The tree is a dictionary whose key `subtasks` holds a list of further dictionaries, so the same function works at every level. The function `book_tree` returns a new tree for given probabilities of the two branches, and `BOOK` keeps the values of the lecture.

**Exercise C.** A task takes 20 s, plus an optional step of 30 s with the probability 0.25 and another of 45 s with the probability 0.1. Store its expected time in `t_c`.

In [ ]:
t_c = None   # your answer

check("Exercise C", t_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_c = expected_time({"time": 20, "subtasks": [{"time": 30, "p": 0.25},
                                             {"time": 45, "p": 0.1}]})

check("Exercise C", t_c)

In [ ]:
# Figure: the expected time of the booking against the share of full days
shares = np.linspace(0, 0.8, 9)
times = [expected_time(book_tree(q_day=q)) for q in shares]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(100 * shares, times, "o-", color="#5A4BB5")
ax.plot([25], [expected_time(BOOK)], "s", color="#C98510", ms=8,
        label="the tree of the lecture")
ax.set_xlabel("bookings that find no free time on the first day (%)")
ax.set_ylabel("expected time of the booking (s)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Python note.** The list comprehension calls `book_tree` once for each share and passes the share by its name, `q_day=q`, while `p_help` keeps its default. The time grows on a straight line, because the branch enters the sum with its probability as a factor.

**Observe and reflect.** How much does each full day cost a patient? Which subtask of the tree would you shorten first, and why?

## 4. The keystroke-level model: Your front desk race &#91;[13](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#ref-13)&#93;

*Lecture: [Predicting task times: The keystroke-level model](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#keystroke-level-model)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "BOOK" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-06/"
               "NB06_users_and_tasks.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The keystroke-level model predicts how long an expert needs for a task from the list of its keystrokes, pointing movements and mental preparations. Python computes the prediction for the two methods of the front desk and compares it with your own times. The comparison shows where the model fits and where a beginner is slower.

**From the formulas to the code.** An operator sequence such as `"MPB"` is a string, and the times of the operators are the dictionary `KLM`. The predicted time $T = \sum_{op} n_{op} t_{op}$ is `klm_time(seq)`.

In [ ]:
def klm_time(seq, times=KLM):
    """Predicted time of an operator sequence such as "MPBMPB"."""
    return round(sum(times[c] for c in seq), 2)

MOUSE = "MPB" + "MPB" + "MPB" + "PBHMKKKK" + "HPB"
KEYS = "MKKKK" + "MK" + "MKKKKK"
print(f"menus with the mouse: {MOUSE} -> {klm_time(MOUSE)} s")
print(f"codes on the keyboard: {KEYS} -> {klm_time(KEYS)} s")
print(f"saving per booking: {klm_time(MOUSE) - klm_time(KEYS):.2f} s")

**Python note.** A string is a sequence of characters, so `for c in seq` visits the operators one by one, and each character looks up its time in the dictionary `KLM`. Spaces would break the lookup, so the sequences are written without them.

**Exercise D.** A method consists of three mental preparations and six keystrokes. Store its predicted time in `t_d`, with the times of `KLM`.

In [ ]:
t_d = None   # your answer

check("Exercise D", t_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_d = 3 * KLM["M"] + 6 * KLM["K"]

check("Exercise D", t_d)

Part D of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_RACE`. Left empty, the variable makes the code use a simulated class of 24 students with assumed times.

In [ ]:
# Paste the results of lab Part D between the triple quotes.
MY_RACE = """
"""

def simulated_race(n_students=24, seed=RANDOM_STATE):
    """Three bookings per method; beginners slower than the model."""
    rng = np.random.default_rng(seed)
    rows = []
    for student in range(1, n_students + 1):
        for method, base in [("mouse", klm_time(MOUSE)), ("keys", klm_time(KEYS))]:
            for task in (1, 2, 3):
                learn = 1.6 - 0.2 * task if method == "keys" else 1.25 - 0.05 * task
                t = base * learn * rng.lognormal(0, 0.18)
                rows.append([student, method, task, round(1000 * t), 0])
    return pd.DataFrame(rows, columns=["student", "method", "task", "time_ms",
                                       "errors"])

if MY_RACE.strip():
    race = pd.read_csv(io.StringIO(MY_RACE.strip()))
    race["student"] = 1
    source = "your own bookings from the interactive lab"
else:
    race = simulated_race()
    source = "a simulated class of 24 students"
print("Data:", source, "|", len(race), "bookings")
measured = race.groupby(["method", "task"])["time_ms"].mean().unstack() / 1000
tasks = list(measured.columns)
measured["model"] = [klm_time(KEYS if m == "keys" else MOUSE)
                     for m in measured.index]
print(measured.round(2))
ratio = measured[tasks].div(measured["model"], axis=0)
print("measured time as a multiple of the model:")
print(ratio.round(2))

**Python note.** `unstack` turns the bookings into columns, so each row holds one method, and `tasks` keeps their numbers. `div(..., axis=0)` divides every row by its own predicted time, which gives the factor by which you were slower or faster than the model.

In [ ]:
# Figure: measured times against the keystroke-level model
fig, ax = plt.subplots(figsize=(7, 3.4))
colours = {"mouse": "#C98510", "keys": "#5A4BB5"}
for m in measured.index:
    ax.plot(tasks, measured.loc[m, tasks], "o-", color=colours[m],
            label=f"{m}, measured")
    ax.axhline(measured.loc[m, "model"], ls="--", color=colours[m],
               label=f"{m}, model")
ax.set_xticks(tasks)
ax.set_xlabel("booking")
ax.set_ylabel("time (s)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** The dashed lines mark the predictions, which do not change from one booking to the next. The measured times usually fall toward them as the methods are learned.

**Observe and reflect.** Which method came closer to the model by the third booking? What does the gap say about predictions of the model for beginners?

## 5. The clinic app, stage 6

*Lecture: [The clinic app, stage 6: Personas, task analysis and predicted task times](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#the-clinic-app-stage-6)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "measured" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-06/"
               "NB06_users_and_tasks.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 6 brings the methods of the week together. The personas come from the groups of Section 2, and each receives operator times of its own. The kiosk booking before and after the redesign of Weeks 3 to 5 becomes a sequence of operators. The saving can then be predicted for each persona and for a year.

In [ ]:
# Operator times of the three personas, assumptions of the stage: a mental
# preparation M and a tap T. The fixed operators are the search of a menu S
# (stage 3), the slide D (stage 4) and the reading of the card R.
PERSONAS = {"Hasan, the frequent visitor": {"M": 2.0, "T": 0.9},
            "Elif, the busy professional": {"M": 1.35, "T": 0.6},
            "Ayten, the carer": {"M": 1.6, "T": 0.7}}
CURRENT = {"ops": "MT" + "MS" + "MT" + "MT" + "M" + "T" * 12 + "MD",
           "fixed": {"S": 5.62, "D": 0.80}}
REDESIGN = {"ops": "R" + "MT" + "MS" + "MTT" + "MTT",
            "fixed": {"S": 3.55, "R": 2.0}}

def op_times(flow, persona):
    """The time of each operator of a flow for one persona."""
    return [persona.get(c, flow["fixed"].get(c)) for c in flow["ops"]]

def kiosk_time(flow, persona):
    """Predicted time of a kiosk flow for one persona."""
    return round(sum(op_times(flow, persona)), 2)

STAGE6 = {}
for name, ops in PERSONAS.items():
    STAGE6[f"{name}, current (s)"] = kiosk_time(CURRENT, ops)
    STAGE6[f"{name}, redesign (s)"] = kiosk_time(REDESIGN, ops)
saving = round(np.mean([kiosk_time(CURRENT, o) - kiosk_time(REDESIGN, o)
                        for o in PERSONAS.values()]), 2)
STAGE6["mean saving per booking (s)"] = saving
STAGE6["hours saved a year"] = round(600 * 250 * saving / 3600, 1)
STAGE6["expected time of the booking tree (s)"] = round(expected_time(BOOK), 1)
for key, value in STAGE6.items():
    print(f"{key:<44} {value}")

**Python note.** A flow is a string of operators with a dictionary of fixed times. `persona.get(c, ...)` looks up the time of `M` and `T` in the persona and falls back on the fixed times for `S`, `D` and `R`. The yearly hours follow the cost formula of Week 1, $C = N D \Delta t$, with the mean saving rounded to two decimals as in the lecture.

In [ ]:
# Figure: predicted times of the personas and the operators of Hasan
names = list(PERSONAS)
x = np.arange(len(names))
cur = [STAGE6[f"{n}, current (s)"] for n in names]
red = [STAGE6[f"{n}, redesign (s)"] for n in names]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(x - 0.2, cur, 0.4, color="#B53F2E", label="current kiosk")
ax[0].bar(x + 0.2, red, 0.4, color="#2E8B57", label="redesigned kiosk")
for xi, a, b in zip(x, cur, red):
    ax[0].text(xi - 0.2, a + 0.4, f"{a:.2f}", ha="center", fontsize=8)
    ax[0].text(xi + 0.2, b + 0.4, f"{b:.2f}", ha="center", fontsize=8)
ax[0].set_xticks(x, [n.split(",")[0] for n in names])
ax[0].set_ylabel("predicted booking time (s)")
ax[0].legend()
COLOURS = {"M": "#5A4BB5", "T": "#C98510", "S": "#0E7A78", "D": "#B53F2E",
           "R": "#585E6E"}
hasan = PERSONAS["Hasan, the frequent visitor"]
for row, flow in enumerate([CURRENT, REDESIGN]):
    left = 0.0
    for c, t in zip(flow["ops"], op_times(flow, hasan)):
        ax[1].barh(row, t, left=left, color=COLOURS[c], edgecolor="white")
        left += t
for c, col in COLOURS.items():
    ax[1].barh(0, 0, color=col, label=c)
ax[1].set_yticks([0, 1], ["current", "redesign"])
ax[1].invert_yaxis()
ax[1].set_xlabel("time for Hasan (s)")
ax[1].legend(ncol=5, fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

**Python note.** `zip` pairs each operator with its time. The variable `left` moves the start of the next bar to the end of the last one, so the bars add up to a timeline. The bars of width 0 only create the entries of the legend.

**Exercise E.** Store the hours that the redesign saves the patients in a year, as kept in `STAGE6`, in `hours_e`.

In [ ]:
hours_e = None   # your answer

check("Exercise E", hours_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
hours_e = STAGE6["hours saved a year"]

check("Exercise E", hours_e)

**Observe and reflect.** Which persona gains the most from the redesign, and which step of the flow causes most of the gain? Which assumption of the stage would you check first with real patients?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE6" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-06/"
               "NB06_users_and_tasks.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Each app below has a frequent task with a current and a redesigned method. The function of Section 4 predicts both times and the saving per year after one change of a name.

The operator sequences and the numbers of uses are illustrative. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "banking"
# choose: clinic, banking, e_government, food_delivery, course_registration
APPS = {"clinic": ("book at the desk", MOUSE, KEYS, 300, 250),
        "banking": ("pay a saved bill", "MPBMPBMPBHMKKKKKKHPB", "MPBMPBMPB",
                    50000, 365),
        "e_government": ("download a document", "MPBMPBMPBMPBMPB", "MPBMPB",
                         20000, 250),
        "food_delivery": ("repeat the last order", "MPBMPBMPBMPBMPBMPB", "MPBMPB",
                          80000, 365),
        "course_registration": ("add a course by its code", "MPBMPBMPBMPB",
                                "MPBHMKKKKKKKK", 5000, 20)}

task, now, new, per_day, days = APPS[APP]
t_now, t_new = klm_time(now), klm_time(new)
print(f"{APP}: {task}")
print(f"  current {now}: {t_now} s | redesign {new}: {t_new} s")
print(f"  saving {t_now - t_new:.2f} s per use, "
      f"{per_day * days * (t_now - t_new) / 3600:.0f} hours a year")

**Python note.** Each entry holds the task, two operator sequences, the uses per day and the days of use in a year. The yearly saving multiplies the saving per use by the uses of a year, as the cost formula of Week 1 does. A saving per use can also be negative, when the redesign is slower.

**Observe and reflect.** Which app saves the most time in a year, and does a large saving per use or a large number of uses drive it? Name one effect on users that the model cannot see.

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write the keystroke-level sequence of committing a change in a version control tool with the menus and with the command line, and compare the predicted times. |
| Web development | Interview three users of a website about one task, code their answers into themes and draw the saturation curve of the three interviews. |
| Mobile development | Build a task tree for ordering a taxi in an app, with a branch for a cancelled ride, and compute the expected time for two shares of cancellations. |
| Game development | Group twenty simulated players by playing time, skill and spending with k-means, and describe a persona for each group. |
| Embedded systems and the Internet of Things | Predict the time to set a programmable thermostat with its buttons and with an app, using the operators of this week. |
| Data science | Repeat the grouping of Section 2 with two and with four groups, and use the within-group sum of squares to argue for a number of personas. |
| Cybersecurity | Model logging in with a password, with a password manager and with a passkey as operator sequences, and compare their predicted times. |
| Health informatics | Add a persona of a nurse who books for patients at the desk, with operator times of its own, and recompute the yearly saving of stage 6. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-06).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>